In [ ]:
import os, glob, subprocess
if not os.path.exists('/kaggle/working/Basma'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/us3ph/Basma.git', '/kaggle/working/Basma'], check=True)
os.chdir('/kaggle/working/Basma'); print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
import os, glob, json, subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
# data-kernel outputs: v2's four builds, the two v3 builds, the two v4 builds and the five v5 OpenFake v2.1 parts
mans = sorted(m for m in glob.glob('/kaggle/input/**/manifest_*.jsonl', recursive=True) if '/basma-data-' in m)
dirs = sorted({os.path.dirname(m) for m in mans}); print('DATA DIRS', dirs)
assert len(dirs) == 13, f'expected cf/recent/reals/eval + v3ai/v3real + v4a/v4b + v5a-e, found {mans}'
print(subprocess.run(['df', '-h', '/tmp', '/kaggle/working'], capture_output=True, text=True).stdout)
# leak guard: training-side images that are near-copies of a Basma Bench or NTIRE test image are left out
excl = sorted(glob.glob('/kaggle/input/**/basma-leak-guard-v5/**/exclude_ids.txt', recursive=True) or glob.glob('/kaggle/input/**/exclude_ids.txt', recursive=True))
assert excl, 'leak guard output (exclude_ids.txt) not mounted'
os.environ['BASMA_EXCLUDE'] = excl[0]; print('BASMA_EXCLUDE', excl[0], sum(1 for _ in open(excl[0])), 'ids')
lg = os.path.join(os.path.dirname(excl[0]), 'LEAKGUARD.md'); os.makedirs('docs', exist_ok=True)
if os.path.exists(lg): subprocess.run(['cp', lg, 'docs/LEAKGUARD.md']); print(open(lg).read())
# baseline scores from v4 (same bench rows and per-image attack seeds): only rows new to v5 are scored
cache = sorted(glob.glob('/kaggle/input/**/runs/kaggle_v4/baselines/thermostatic.jsonl', recursive=True))
os.environ['BASELINE_CACHE'] = os.path.dirname(cache[0]) if cache else ''; print('BASELINE_CACHE', os.environ['BASELINE_CACHE'] or 'none')
cfg = json.load(open('configs/kaggle_2xT4_v5.json')); cfg['data_dirs'] = dirs
json.dump(cfg, open('configs/kaggle_run.json', 'w'), indent=1); print(json.dumps(cfg, indent=1))


In [ ]:
!bash scripts/kaggle_train.sh configs/kaggle_run.json


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('docs/RESULTS.md').read()))
if os.path.exists('runs/kaggle_v5/ntire/NTIRE.md'): display(Markdown(open('runs/kaggle_v5/ntire/NTIRE.md').read()))
